# Autoresearch-Stroke: Experiment Analysis

Analysis of autonomous hyperparameter tuning results for stroke artery classification (ACA/MCA/PCA).
Primary metric: **Macro F1** (higher is better).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")
for col in ['macro_f1', 'aca_f1', 'mca_f1', 'pca_f1', 'memory_mb']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df['epochs'] = pd.to_numeric(df['epochs'], errors='coerce')
df['status'] = df['status'].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df['status'].value_counts()
print('Experiment outcomes:')
print(counts.to_string())

n_keep = counts.get('KEEP', 0)
n_discard = counts.get('DISCARD', 0)
n_crash = counts.get('CRASH', 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f'\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}')

In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
print(f'KEPT experiments ({len(kept)} total):\n')
for i, row in kept.iterrows():
    print(f"  #{i:3d}  F1={row['macro_f1']:.6f}  ACA={row['aca_f1']:.4f}  "
          f"MCA={row['mca_f1']:.4f}  PCA={row['pca_f1']:.4f}  {row['description']}")

## Macro F1 Progress Over Time

Track how the best (kept) macro_f1 evolves. The running maximum shows the frontier.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df['status'] != 'CRASH'].copy().reset_index(drop=True)
baseline_f1 = valid.loc[0, 'macro_f1']

# Discarded as faint dots
disc = valid[valid['status'] == 'DISCARD']
ax.scatter(disc.index, disc['macro_f1'], c='#cccccc', s=12, alpha=0.5, zorder=2, label='Discarded')

# Kept experiments as prominent dots
kept_v = valid[valid['status'] == 'KEEP']
ax.scatter(kept_v.index, kept_v['macro_f1'], c='#2ecc71', s=50, zorder=4,
           label='Kept', edgecolors='black', linewidths=0.5)

# Running maximum step line
kept_mask = valid['status'] == 'KEEP'
kept_idx = valid.index[kept_mask]
kept_f1 = valid.loc[kept_mask, 'macro_f1']
running_max = kept_f1.cummax()
ax.step(kept_idx, running_max, where='post', color='#27ae60',
        linewidth=2, alpha=0.7, zorder=3, label='Running best')

# Label each kept experiment
for idx, f1 in zip(kept_idx, kept_f1):
    desc = str(valid.loc[idx, 'description']).strip()
    if len(desc) > 40: desc = desc[:37] + '...'
    ax.annotate(desc, (idx, f1), textcoords='offset points',
                xytext=(6, 6), fontsize=7.5, color='#1a7a3a',
                alpha=0.9, rotation=25, ha='left', va='bottom')

# Target line
ax.axhline(y=0.87, color='red', linestyle='--', alpha=0.5, label='Target (0.87)')

best = kept_f1.max() if len(kept_f1) > 0 else baseline_f1
ax.set_xlabel('Experiment #', fontsize=12)
ax.set_ylabel('Macro F1 (higher is better)', fontsize=12)
ax.set_title(f'Autoresearch-Stroke Progress: {len(df)} Experiments, {len(kept_v)} Improvements', fontsize=14)
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.2)

margin = (best - baseline_f1) * 0.3 if best > baseline_f1 else 0.01
ax.set_ylim(baseline_f1 - margin, max(best + margin, 0.875))

plt.tight_layout()
plt.savefig('progress.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to progress.png')

## Per-Class F1 Analysis

Track ACA, MCA, PCA F1 scores and the seesaw effect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Per-class F1 over experiments (kept only)
kept = df[df['status'] == 'KEEP'].reset_index()
if len(kept) > 0:
    ax = axes[0]
    x = range(len(kept))
    ax.plot(x, kept['aca_f1'], 'o-', color='#e74c3c', label='ACA F1', markersize=6)
    ax.plot(x, kept['mca_f1'], 's-', color='#3498db', label='MCA F1', markersize=6)
    ax.plot(x, kept['pca_f1'], '^-', color='#2ecc71', label='PCA F1', markersize=6)
    ax.plot(x, kept['macro_f1'], 'D-', color='#9b59b6', label='Macro F1', markersize=6, linewidth=2)
    ax.axhline(y=0.80, color='#e74c3c', linestyle=':', alpha=0.3, label='ACA target (0.80)')
    ax.axhline(y=0.90, color='#3498db', linestyle=':', alpha=0.3, label='MCA target (0.90)')
    ax.axhline(y=0.85, color='#2ecc71', linestyle=':', alpha=0.3, label='PCA target (0.85)')
    ax.set_xlabel('Kept Experiment #')
    ax.set_ylabel('F1 Score')
    ax.set_title('Per-Class F1 Progression (Kept Experiments)')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.2)

    # Seesaw plot: ACA F1 vs MCA F1
    ax2 = axes[1]
    all_valid = df[df['status'] != 'CRASH']
    ax2.scatter(all_valid['aca_f1'], all_valid['mca_f1'], c='#cccccc', s=20, alpha=0.5, label='All')
    ax2.scatter(kept['aca_f1'], kept['mca_f1'], c='#2ecc71', s=50, edgecolors='black',
               linewidths=0.5, zorder=3, label='Kept')
    ax2.axvline(x=0.80, color='#e74c3c', linestyle=':', alpha=0.3)
    ax2.axhline(y=0.90, color='#3498db', linestyle=':', alpha=0.3)
    ax2.set_xlabel('ACA F1')
    ax2.set_ylabel('MCA F1')
    ax2.set_title('Seesaw Effect: ACA F1 vs MCA F1')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('perclass.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary Statistics

In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
if len(kept) > 0:
    baseline_f1 = kept.iloc[0]['macro_f1']
    best_f1 = kept['macro_f1'].max()
    best_row = kept.loc[kept['macro_f1'].idxmax()]

    print(f'Baseline macro_f1:  {baseline_f1:.6f}')
    print(f'Best macro_f1:      {best_f1:.6f}')
    print(f'Total improvement:  {best_f1 - baseline_f1:+.6f} ({(best_f1 - baseline_f1) / baseline_f1 * 100:+.2f}%)')
    print(f'Best experiment:    {best_row["description"]}')
    print(f'Best per-class:     ACA={best_row["aca_f1"]:.4f}  MCA={best_row["mca_f1"]:.4f}  PCA={best_row["pca_f1"]:.4f}')
    print()

    # Target check
    print('Target Status:')
    targets = {
        'Macro F1 >= 0.87': best_f1 >= 0.87,
        'ACA F1 >= 0.80': kept['aca_f1'].max() >= 0.80,
        'MCA F1 >= 0.90': kept['mca_f1'].max() >= 0.90,
        'PCA F1 >= 0.85': kept['pca_f1'].max() >= 0.85,
    }
    for name, met in targets.items():
        print(f"  {name}: {'REACHED' if met else 'NOT YET'}")
else:
    print('No kept experiments yet.')

## Top Improvements (Ranked by Delta)

In [ ]:
kept = df[df['status'] == 'KEEP'].copy()
if len(kept) > 1:
    kept['prev_f1'] = kept['macro_f1'].shift(1)
    kept['delta'] = kept['macro_f1'] - kept['prev_f1']
    hits = kept.iloc[1:].sort_values('delta', ascending=False)

    print(f"{'Rank':>4}  {'Delta':>8}  {'F1':>10}  Description")
    print('-' * 70)
    for rank, (_, row) in enumerate(hits.iterrows(), 1):
        print(f"{rank:4d}  {row['delta']:+.6f}  {row['macro_f1']:.6f}  {row['description']}")
    print(f"\n{'':>4}  {hits['delta'].sum():+.6f}  {'':>10}  TOTAL improvement over baseline")
else:
    print('Need at least 2 kept experiments to show deltas.')